In [140]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Import library

In [1]:
import pandas as pd
import numpy as np
import regex as re
import networkx as nx
import itertools as it
import matplotlib.pyplot as plt
import pickle
import os
from sentence_transformers import SentenceTransformer, util
import torch
import seaborn as sns 
import nltk

# Read data


In [2]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
os.chdir(parent_dir)

In [3]:

my_data = {}
folder_path = 'Output/Overcome/'
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rb') as f:
            print("load:",filename)
            data = pickle.load(f)
            # store the loaded data in a dictionary for further processing, the key is the file name, the dictionary is data
            my_data[filename] = data
                        
        

load: Overcome SF3 - 5- 10 - 2023.json
load: Overcome RC4 6-7-2023.json
load: SF2-4-28-2023.json
load: SF1-4-26-2023 revised.json
load: SF5-5-18-2023.json
load: Overcome RC1 4-5-2023.json
load: overcome RC2 4-5-2023.json
load: Ocome SF3 -5 10 23 revised.json
load: Overcome 5 6-7-2023.json
load: SF1-4-26-2023.json
load: Overcome SF4 - 5 - 10 - 2023.json
load: overcome RC3 5-5-2023.json


In [4]:
# rename the files, to read better. 
        
rc_rename_files = {  
    "Overcome RC4 6-7-2023.json":  "RC4",
    "Overcome RC1 4-5-2023.json": "RC1" ,
    "overcome RC2 4-5-2023.json":"RC2",
    "Overcome 5 6-7-2023.json": "RC5", 
    "overcome RC3 5-5-2023.json": "RC3", 
    "Overcome SF3 - 5- 10 - 2023.json":"SF3", 
    "SF2-4-28-2023.json":"SF2", 
    "SF1-4-26-2023 revised.json":"SF1", 
    "SF5-5-18-2023.json":"SF5", 
    "Ocome SF3 -5 10 23 revised.json":"SF3",
    "SF1-4-26-2023.json":"SF1",
    "Overcome SF4 - 5 - 10 - 2023.json":"SF4"

}

my_data = {rc_rename_files.get(key, key): value for key, value in my_data.items()}

my_data.keys()

dict_keys(['SF3', 'RC4', 'SF2', 'SF1', 'SF5', 'RC1', 'RC2', 'RC5', 'SF4', 'RC3'])

# Get Text similarity


In [5]:
# Load sentence transformer model
model1 = SentenceTransformer('paraphrase-MiniLM-L6-v2')


/Users/gl665/PycharmProjects/Phrame-MentalMaps/.new_venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
df_cld = pd.DataFrame()
# create column of dataset, each cell is the keys from my_data dictionary
df_cld['dataset'] = pd.Series(list(my_data.keys()))
df_cld['vars'] = df_cld['dataset'].apply(lambda x: list(my_data[x][0].values()))
df_cld['embeddings'] = df_cld['vars'].apply(lambda x: model.encode([x], convert_to_tensor=True)[0])
df_cld.head(3)

NameError: name 'model' is not defined

## Cosine Similarity 

Using Cosine Simil arity to calculate the similarity between the embeddings of the variables in the datasets. 
The result shows that most datasets have some similarity, indicating that there may be some overlap in the variables used across the datasets.



In [ ]:
similarity_matrix = pd.DataFrame(index=df_cld.index, columns=df_cld.index)

for i in range(0, len(my_data)): 
    for j in range(0, len(my_data)):
        if i != j:
            embedding1 = df_cld['embeddings'].iloc[i].unsqueeze(0)
            embedding2 = df_cld['embeddings'].iloc[j].unsqueeze(0)
            
            similarity_score = torch.nn.functional.cosine_similarity(embedding1, embedding2)
            # store the score in the similarity matrix
            similarity_matrix.loc[i, j] = similarity_score.item()

similarity_matrix.columns = df_cld['dataset']
similarity_matrix.index = df_cld['dataset']
similarity_matrix

In [ ]:
similarity_matrix = similarity_matrix.astype(float)
sns.heatmap(similarity_matrix, annot=True)

In [ ]:
# df_cld['cld_vars'] = df_cld['vars'].apply(listToString)

In [ ]:

# from sklearn.feature_extraction.text import CountVectorizer
# vectorizer = CountVectorizer()
# vectorizer.fit(list(df_cld['cld_vars']))
# print(vectorizer.vocabulary_)

In [ ]:
string_list = []

for i in my_data: 
    string_list.append(', '.join(list(my_data[i][0].values())))
    
string_list
    

## Term Frequency Inverse Document Frequency (TFIDF). 
TFIdf way to measure how unique a certain word is relative to every other word in a document. This is calculated on a scale from 0–1 with the most common words approaching 0 and the most unique words approaching 1. 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(string_list)
arr = X.toarray()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def create_heatmap(similarity, cmap = "YlGnBu"):
  df = pd.DataFrame(similarity)
  mask = np.tril(np.ones(df.shape, dtype=bool))

  df.columns = df_cld['dataset']
  df.index = df_cld['dataset']
  fig, ax = plt.subplots(figsize=(5,5))
  sns.heatmap(df, cmap=cmap, annot=True, ax=ax, mask = mask)

In [ ]:
create_heatmap(cosine_similarity(arr))

# Using pre-trained sentence transformer to compute text similarity

Sentence transformer models captures semantic information in text and can be used to compute text similarity. Here, we use the 'universal-sentence-encoder' model to compute the similarity between the variables in the datasets.

In [ ]:
from absl import logging
import tensorflow as tf
import tensorflow_hub as hub

module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
model = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return model(input)


In [ ]:
message_embeddings_ = embed(string_list)

corr = np.inner(message_embeddings_, message_embeddings_)

In [ ]:
create_heatmap(corr)

# Cluster Analysis

https://stackoverflow.com/questions/73183103/clustering-based-on-semantic-similarity-returning-no-values


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
from kmeans_pytorch import kmeans
import torch

def detect_clusters(X, nclusters, tol=1e-6):
  X = torch.as_tensor(X)
  assert X.ndim == 2
  # Project the points in a hypersphere
  X1 = X / torch.sqrt(torch.sum(X**2, axis=-1, keepdims=True))

  # Run kmeans on the normalized points with euclidean distance
  cluster_ID, C = kmeans(X1, nclusters, distance='euclidean', tol=tol)
  return cluster_ID, C

In [ ]:
embeddings = model1.encode(string_list, show_progress_bar=True, convert_to_numpy=True)
embeddings

In [ ]:
X =  torch.as_tensor(embeddings)
label, center = detect_clusters(X, 2)
# Create a dataframe to store the cluster labels
df_cluster = pd.DataFrame({'cluster': label, 'label': list(my_data.keys())})

for c, s in zip(label, my_data.keys()):
    print(f'[{c}] {s}')
    # store in dataframe 

In [ ]:
df_cluster

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X[:,0], X[:,1])
colors = ['r', 'g']
plt.title("Cluster of OverCome")
for i, txt in enumerate(df_cluster['label']):
    ax.annotate(txt, (X[i,0], X[i,1]))
    # get different color based on cluster label in df_cluster
    color = colors[df_cluster['cluster'][i]]
    
    ax.scatter(X[i,0], X[i,1], c=color)

# Single variable name clustering 


In [ ]:
# Get embeddings of each variable in each text 

# Extract variable names and store them in a dictionary with paragraph indices
global_variable_dict = {}
def get_single_var_dictionary(text, index):
    for variables in text:
        var_names = [var.strip() for var in variables.split(',')]
        for var in var_names:
            variable_dict[var] = index
    return variable_dict

for i, text in zip(df_cld['dataset'],df_cld['vars']):
    variable_dict = get_single_var_dictionary(text, i)
    global_variable_dict.update(variable_dict)

In [ ]:
global_variable_dict

In [ ]:
# def embed(input):
#     return model(input)
def get_embed(input): 
    return model1.encode(input, show_progress_bar=False, convert_to_numpy=False)
    
# get all the embeddings in the key of glo bal_variable_dict
var_embeddings = {}
for var, keys in global_variable_dict.items():
    embeddings = get_embed(var)
    # embeddings = np.ndarray.flatten(embeddings)
    var_embeddings[var] = embeddings
    
    

In [ ]:
# Ensure all dictionary values are PyTorch tensors
for var, embedding in var_embeddings.items():
    if not isinstance(embedding, torch.Tensor):
        raise TypeError(f"Expected PyTorch tensor, but got {type(embedding)} for variable {var}")


In [ ]:
var_embeddings

In [ ]:
X =  torch.as_tensor(embeddings)
label, center = detect_clusters(X, 5)
# Create a dataframe to store the cluster labels
df_cluster = pd.DataFrame({'cluster': label, 'label': list(global_variable_dict.values()), 'cld variable': list(global_variable_dict.keys())})

for c, s in zip(label, my_data.keys()):
    print(f'[{c}] {s}')
    # store in dataframe 

In [ ]:
df_cluster

In [ ]:
fig, ax = plt.subplots()
# Figsize can be adjusted to control the size of the plot
ax.figure.set_size_inches(20, 20)
ax.scatter(X[:,0], X[:,1])
colors = ['r', 'g', 'b', 'c', 'm']
plt.title("Cluster of OverCome")
for i, txt in enumerate(df_cluster['label']):
    ax.annotate(txt, (X[i,0], X[i,1]))
    # get different color based on cluster label in df_cluster
    color = colors[df_cluster['cluster'][i]]
    
    ax.scatter(X[i,0], X[i,1], c=color)

In [7]:
from sklearn.cluster import KMeans

# # Convert embeddings to a tensor
# X =  torch.as_tensor(var_embeddings.values())
# label, center = detect_clusters(X, 2)
# # Create a dataframe to store the cluster labels
# df_cluster = pd.DataFrame({'cluster': label, 'label': list(my_data.keys())})
# 
# for c, s in zip(label, my_data.keys()):
#     print(f'[{c}] {s}')
#     
# 
# Cluster the embeddings
num_clusters = 5  # Adjust the number of clusters as needed
kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(X)
y_kmeans = kmeans.predict(X)

# Print clusters
clusters = {}
for idx, label in enumerate(kmeans.labels_):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append(list(global_variable_dict.keys())[idx])

# for cluster, vars in clusters.items():
#     print(f"Cluster {cluster}: {', '.join(vars)}")

NameError: name 'X' is not defined

In [ ]:
# kmeans
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, s=50, cmap='viridis')

# plt.plot(X[:, 0], X[:, 1], c= y_kmeans, markersize=2,s=50, cmap='viridis')
# Plot the centroids as a white X
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, alpha=0.5);

# plt.scatter(
#     centroids[:, 0],
#     centroids[:, 1],
#     marker="x",
#     s=169,
#     linewidths=3,
#     color="w",
#     zorder=10,
# )


In [8]:
centers = kmeans.cluster_centers_


NameError: name 'kmeans' is not defined